# Descriptive analysis — PWR workforce elasticity (T5)

**Scope.** Provider × financial-year panel covering NHS trusts 2021/22 to
2024/25 (audited TAC vintages) plus operational outcome controls from
HCHS Workforce Statistics, NHS Vacancy Statistics, the Monthly A&E Time
Series, and the RTT February 2026 full extract.

**Adaptation note.** TAC publishes staff cost as Permanent (substantive)
vs Other staff (Bank + Agency + Contract for Services combined) — the
Bank-versus-Agency split is not in the consolidated dataset. The
descriptive analysis here therefore inspects `other_staff_pay_gbp` and
its ratios. Bank-vs-Agency shift-cost case studies are reported
separately from the REC FOI extracts in `data/rec_foi/`.

**Run-time.** End-to-end execution should take well under five minutes
on a laptop. Figures are written to `outputs/figures/descriptive/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pwr_elasticity import features, io, panel

DATA = Path("../data")
FIGURES = Path("../outputs/figures/descriptive")
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="notebook", style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 160


## 1. Load and assemble

In [ ]:
tac = io.read_tac(DATA / "tac_provider_accounts")
hchs = io.read_hchs_staff_in_post(
    DATA / "workforce_stats" / "hchs_oct_2025" /
    "Core 1. Staff group - England, NHSE region, ICS and org, Oct-25.csv"
)
turnover = io.read_hchs_turnover(
    DATA / "workforce_stats" / "turnover_oct_2025" /
    "Turnover 1. Staff group - NHSE region, annual, Sep-09 to Oct-25.csv"
)
vacancies = io.read_vacancies(
    DATA / "vacancy_stats" / "nhs-vac-stats-apr15-dec25-eng-tables.xlsx"
)
ae = io.read_ae(DATA / "ae_performance" / "Monthly-AE-Time-Series-March-2026.xls")
rtt = io.read_rtt(
    DATA / "rtt_performance" / "feb26" / "20260228-RTT-February-2026-full-extract.csv"
)
ods = io.read_ods_trusts(DATA / "reference" / "etr-nhs-trusts.csv")

p = panel.build_panel(tac, hchs, turnover, vacancies, ae, rtt, ods)
f = features.compute_features(p)
exclusions = panel.provider_exclusions({"tac": tac, "hchs": hchs, "ods": ods})

print(f"panel shape: {p.shape}")
print(f"feature shape: {f.shape}")
print(f"providers: {p['org_code'].nunique()}")
print(f"financial years: {sorted(p['financial_year'].unique())}")
print(f"exclusions: {len(exclusions)} rows")


## 2. Outcome trajectory — Other-staff pay over time

Headline descriptive plot: median `other_staff_pay_gbp` per FY across
all providers in the panel.

In [ ]:
summary = (
    f.groupby("financial_year")[["other_staff_pay_gbp", "substantive_pay_gbp",
                                  "total_pay_gbp", "pay_intensity",
                                  "other_to_substantive_ratio"]]
    .median()
    .round(2)
)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
medians = f.groupby("financial_year")["pay_intensity"].median()
medians.plot(marker="o", ax=ax, color="C0")
ax.set_xlabel("Financial year")
ax.set_ylabel("Other-staff GBP per substantive FTE")
ax.set_title("Median pay intensity by financial year")
fig.tight_layout()
fig.savefig(FIGURES / "01_pay_intensity_by_fy.png")
plt.show()


## 3. Distribution of `other_to_substantive_ratio` by year and provider type

Boxplots show the within-FY spread of the substitution measure across
providers; the FY-level means quantify the policy-period direction.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=f,
    x="financial_year",
    y="other_to_substantive_ratio",
    hue="provider_type",
    ax=ax,
)
ax.set_ylabel("Other-staff GBP / Substantive GBP")
ax.set_xlabel("Financial year")
ax.set_title("Other-to-substantive ratio by FY and provider type")
ax.legend(title="Provider type", loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "02_other_to_substantive_ratio_box.png")
plt.show()


## 4. REC case-study trusts

REC FOI disclosures (January and May 2026) cited four trusts as the
empirical anchor for the bank-vs-agency cost-inversion narrative:
Nottingham University Hospitals, Imperial College Healthcare,
Manchester University, and Newcastle upon Tyne Hospitals. Most are
Foundation Trusts and may therefore not appear in the published
NHS-trust TAC dataset (Foundation Trusts publish to a separate
TAC dataset that is not yet loaded). Where the row is missing, that
finding is itself informative.

In [ ]:
case_studies = [
    "Nottingham University Hospitals",
    "Imperial College Healthcare",
    "Manchester University",
    "Newcastle upon Tyne Hospitals",
]
mask = f["org_name"].astype("string").str.lower().str.startswith(
    tuple(name.lower() for name in case_studies)
)
hits = f.loc[mask, ["org_code", "org_name", "financial_year",
                    "substantive_pay_gbp", "other_staff_pay_gbp",
                    "other_to_substantive_ratio"]].sort_values(
    ["org_name", "financial_year"]
)
print(f"REC case-study rows in panel: {len(hits)} (of {len(case_studies)} named trusts)")
hits


## 5. Operational pressure — vacancy rate, A&E, RTT

Descriptive context for the controls that enter the TWFE regression in
T6. Vacancy is a region × FY average broadcast to providers; A&E is the
England aggregate broadcast; RTT is a FY-end stock and is therefore
only populated where the panel year has an in-FY RTT extract.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
f.groupby("financial_year")["vacancy_rate"].mean().plot(
    marker="o", ax=axes[0], color="C1"
)
axes[0].set_title("Mean vacancy rate")
axes[0].set_ylabel("Fraction")
f.groupby("financial_year")["pct_met_4hr"].mean().plot(
    marker="o", ax=axes[1], color="C2"
)
axes[1].set_title("A&E 4-hour standard")
axes[1].set_ylabel("Fraction met")
f.groupby("financial_year")["turnover_rate"].mean().plot(
    marker="o", ax=axes[2], color="C3"
)
axes[2].set_title("Mean turnover rate")
axes[2].set_ylabel("Fraction")
for ax in axes:
    ax.set_xlabel("Financial year")
fig.tight_layout()
fig.savefig(FIGURES / "03_operational_pressure.png")
plt.show()


## 6. Pairwise correlations of candidate model covariates

In [ ]:
candidate_cols = [
    "log_other_staff_pay",
    "log_substantive_pay",
    "pay_intensity",
    "other_to_substantive_ratio",
    "policy_intensity_t",
    "vacancy_rate",
    "turnover_rate",
    "pct_met_4hr",
    "staff_in_post_fte",
]
corr = f[candidate_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax,
            vmin=-1, vmax=1)
ax.set_title("Pairwise Pearson correlations")
fig.tight_layout()
fig.savefig(FIGURES / "04_correlation_heatmap.png")
plt.show()

corr.round(3)


## 7. Exclusions and outliers

The `provider_exclusions` log records merger transitions, missing-HCHS
rows and ICS-reorganisation candidates. Outliers on
`other_to_substantive_ratio` (> 3 × interquartile range above the upper
hinge) are also tabulated; T4 features keeps them, but T6 robustness
will re-fit with them removed.

In [ ]:
exclusions.head(20)


In [ ]:
ratio = f["other_to_substantive_ratio"].dropna().astype("float64")
q1, q3 = np.percentile(ratio, [25, 75])
upper = q3 + 3.0 * (q3 - q1)
outliers = f[f["other_to_substantive_ratio"].astype("Float64") > upper][
    ["org_code", "org_name", "financial_year", "other_to_substantive_ratio"]
].sort_values("other_to_substantive_ratio", ascending=False)
print(f"3*IQR upper hinge: {upper:.3f}")
print(f"outliers above hinge: {len(outliers)}")
outliers.head(10)


## 8. Headline take-aways for the T6 specification

1. **Direction of effect.** Median pay intensity (other-staff GBP per
   substantive FTE) declines monotonically from 2022/23 onwards,
   consistent with the agency-rule tightening regime. The TWFE
   coefficient on `policy_intensity_t` should therefore be negative.
2. **Heterogeneity.** Box-plots by `provider_type` show the largest
   variance among acute providers; specialist and ambulance categories
   sit at distinct levels and may warrant stratified estimation in T6c.
3. **Multicollinearity risk.** `log_substantive_pay` and
   `log_other_staff_pay` co-move closely; the regression should
   include both with care (T7 VIF check is the safety net).
4. **REC case studies.** Foundation-Trust coverage gap in the TAC panel
   means the Imperial / Manchester / Newcastle benchmarks cannot be
   produced from open data alone; T8 dashboard will need to combine
   the panel result with the REC FOI shift-cost extracts.
